# NOTEBOOK PARA FUNCIÓN COMPLETA DE ANÁLISIS DE MÁRGENES

## La función ya se encuentra dividida en varios archivos .py dentro de la carpeta Análisis márgenes. Lo que se hará en este notebook es unir todos los pasos en un solo archivo para poder realizar los análisis con mayor velocidad
## Las funciones que se utilizarán quedarán guardadas de ahora en adelante en el pipeline, dentro de la carpeta procesamiento, en el archivo .py llamado funciones_para_analisis_margen

In [416]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
import re
import os
from pipeline.procesamiento.funciones_para_analisis_margen import (extraer_numero_de_paquetes,
                                                                   convertir_fechas,
                                                                   limpiar_sku,
                                                                   obtener_proveedores,
                                                                   clasificar_estado,
                                                                   expand_sku,
                                                                   aplicar_descuentos,
                                                                   descuentos_importadoras,
                                                                   descuento_sku_prefijo,
                                                                   calcular_costo_final,
                                                                   detectar_sku_faltante,
                                                                   clasificar_precio)

In [417]:
año = 2025 # int(input('Indica el año (ejemplo: 2024, 2025): '))
cuenta_meli = input('Indica la cuenta de Mercado Libre a la que corresponde este análisis (ejemplo: BLACKPARTS): ')
fecha = "AGOSTO 2025" # input('Indica la fecha del análisis (ejemplo: JUNIO 2025): ')
fecha_costos = "2025-09-01" # input('Indica la fecha a la que están los costos (formato YYYY-MM-DD): ')

ACTIVAR_DESCUENTOS = False  # FALSE: no se aplican descuentos | TRUE: se aplican descuentos

In [418]:
# Paso 1: Leer DataFrame y eliminar paquetes

archivo_venta = f'/Users/martincarrasco/Desktop/Martín_Carrasco/Reportes/{año}/Cuentas RDS/{cuenta_meli}/VENTAS {cuenta_meli} {fecha}.xlsx'
hoja_venta = 'Ventas CL'
skiprows = 5
df = pd.read_excel(archivo_venta, sheet_name = hoja_venta, skiprows = skiprows, dtype = {"# de venta": str})
print(f"Existen {df['# de venta'].nunique()} registros en la base de datos de ventas")

if 'Fecha de venta' in df.columns:
    df['Fecha de venta'] = df['Fecha de venta'].apply(convertir_fechas)

df['Forma de entrega'] = df['Forma de entrega'].replace(r'^\s*$', np.nan, regex=True)
df["Forma de entrega"] = df["Forma de entrega"].fillna(method = 'ffill')

df_ingresos = df[["# de venta", "Fecha de venta", "Estado", "Unidades", "SKU", "# de publicación", "Título de la publicación", "Precio unitario de venta de la publicación (CLP)", "Forma de entrega"]]
df_ingresos["Cuenta Meli"] = cuenta_meli

wb = load_workbook(archivo_venta, data_only = True)
ws = wb[hoja_venta] if hoja_venta else wb.active
idx_estado = list(df.columns).index('Estado')
estados_backrounds = []
encabezado_fila_excel = skiprows + 1
for row in ws.iter_rows(min_row = encabezado_fila_excel + 1, max_row = ws.max_row):
    cell = row[idx_estado]
    fill = cell.fill
    color = fill.fgColor.rgb if fill and fill.fgColor and fill.fgColor.type == "rgb" else None
    estados_backrounds.append(color)

paquete_indices_todos = []
encabezados_indices = []
i = 0
while i < len(df):
    fondo_actual = estados_backrounds[i]
    estado_valor = str(df.iloc[i]['Estado'])
    if fondo_actual and fondo_actual != "00000000" and "Paquete de" in estado_valor:
        n_items = extraer_numero_de_paquetes(estado_valor)
        rango_paquete = list(range(i, i + n_items + 1))
        paquete_indices_todos.extend(rango_paquete)
        encabezados_indices.append(i)
        i += n_items + 1
    else:
        i += 1

print(f'Hubo {len(encabezados_indices)} ventas en paquete')

df_encabezados = pd.read_excel(archivo_venta, sheet_name = hoja_venta, skiprows = skiprows, dtype = {"# de venta": str})
df_encabezados = df_encabezados.iloc[encabezados_indices].copy()
df_paquetes = df_encabezados[["# de venta", "Fecha de venta", "Ingresos por productos (CLP)"]]
df_paquetes['Cuenta Meli'] = cuenta_meli
df_paquetes['Ingresos por productos (CLP) Neto'] = df_paquetes['Ingresos por productos (CLP)'] / 1.19
if 'Fecha de venta' in df_paquetes.columns:
    df_paquetes['Fecha de venta'] = df_paquetes['Fecha de venta'].apply(convertir_fechas)

ingreso_total = df['Ingresos por productos (CLP)'].sum()
ingreso_paquetes = df.iloc[encabezados_indices]['Ingresos por productos (CLP)'].sum()
porcentaje = 100 * ingreso_paquetes / ingreso_total if ingreso_total else 0

print(f'Las ventas en paquete representan el {porcentaje:.2f}% de las ventas')

df = df.drop(index = paquete_indices_todos).reset_index(drop = True)

df['# de venta'] = df['# de venta'].astype(str)

df['Cuenta Meli'] = cuenta_meli

df = df[df["Ingresos por productos (CLP)"].notna()]

print(f"Existen {df['# de venta'].nunique()} registros en la base de datos de ventas sin contar las ventas en paquete, y con un ingreso válido")

Existen 365 registros en la base de datos de ventas


/var/folders/lr/rkgxn4r50h712qndyykwjmfh0000gp/T/ipykernel_31224/2565330500.py:13: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["Forma de entrega"] = df["Forma de entrega"].fillna(method = 'ffill')
/var/folders/lr/rkgxn4r50h712qndyykwjmfh0000gp/T/ipykernel_31224/2565330500.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ingresos["Cuenta Meli"] = cuenta_meli


Hubo 14 ventas en paquete
Las ventas en paquete representan el 5.57% de las ventas
Existen 323 registros en la base de datos de ventas sin contar las ventas en paquete, y con un ingreso válido


/var/folders/lr/rkgxn4r50h712qndyykwjmfh0000gp/T/ipykernel_31224/2565330500.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_paquetes['Cuenta Meli'] = cuenta_meli
/var/folders/lr/rkgxn4r50h712qndyykwjmfh0000gp/T/ipykernel_31224/2565330500.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_paquetes['Ingresos por productos (CLP) Neto'] = df_paquetes['Ingresos por productos (CLP)'] / 1.19
/var/folders/lr/rkgxn4r50h712qndyykwjmfh0000gp/T/ipykernel_31224/2565330500.py:52: SettingWithCopyWarning: 
A v

In [419]:
# Del paso 1 se obtienen las ventas en paquete:

output_folder = f'/Users/martincarrasco/Desktop/Martín_Carrasco/Análisis márgenes/Data Márgenes/{año}/{fecha}'
os.makedirs(output_folder, exist_ok = True)

output_path_paquetes = f'{output_folder}/{cuenta_meli} {fecha} Ventas en paquete.xlsx'
df_paquetes.to_excel(output_path_paquetes, index = False)

In [420]:
# Paso 2: Editar base de datos para análisis de ingresos (df_ingresos)

# df_ingresos = df_ingresos[(df_ingresos['SKU'].notna()) & (df_ingresos['SKU'] != "")]

df_ingresos = df_ingresos[~df_ingresos["Estado"].str.match(r"Paquete de [2-9]\d* productos", na=False)]

df_ingresos['SKU_MAYUSC'] = (
    df['SKU']
    .astype(str)
    .str.upper()
    .str.strip()
)

# df_ingresos = df_ingresos[(df_ingresos['SKU_MAYUSC'].notna()) & (df_ingresos['SKU_MAYUSC'] != "")]

df_ingresos['SKU_limpio'] = df_ingresos['SKU'].apply(limpiar_sku)

df_ingresos['Proveedor_siglas'] = df_ingresos['SKU_limpio'].apply(obtener_proveedores)
df_ingresos['Proveedor único'] = df_ingresos['Proveedor_siglas'].apply(lambda x: 'Sí' if len(x) == 1 else 'No')
df_ingresos['Proveedor'] = df_ingresos['Proveedor_siglas'].apply(lambda x: " / ".join(x))
df_ingresos = df_ingresos.drop(columns = ['Proveedor_siglas'])

df_ingresos = df_ingresos[df_ingresos["Precio unitario de venta de la publicación (CLP)"].notna() & (df_ingresos["Precio unitario de venta de la publicación (CLP)"] != 0)]
df_ingresos['Ingresos por venta (CLP)'] = df_ingresos['Unidades'] * df_ingresos['Precio unitario de venta de la publicación (CLP)']
df_ingresos['Ingresos por venta (CLP) Neto'] = df_ingresos['Ingresos por venta (CLP)'] / 1.19

df_ingresos['Clasificación Estado'] = df_ingresos['Estado'].apply(clasificar_estado)

In [421]:
# Del paso 2 se obtienen los ingresos totales (ventas unitarias + todas las ventas que componen las ventas en paquete)

output_path_ingresos_totales = f'{output_folder}/{cuenta_meli} {fecha} Ingresos totales.xlsx'
df_ingresos.to_excel(output_path_ingresos_totales, index = False)

In [422]:
# Paso 3: Limpieza de SKU y obtención de proveedores (tal como se hizo en df_ingresos)

df['Clasificación Estado'] = df_ingresos['Estado'].apply(clasificar_estado)

df["SKU_MAYUSC"] = (
    df["SKU"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df = df[df["SKU_MAYUSC"].notna() & (df["SKU_MAYUSC"] != "")]
df["SKU_limpio"] = df["SKU"].apply(limpiar_sku)

df["Proveedor_siglas"] = df["SKU_limpio"].apply(obtener_proveedores)
df["Proveedor único"] = df["Proveedor_siglas"].apply(lambda x: "Sí" if len(x) == 1 else "No")
df["Proveedor"] = df["Proveedor_siglas"].apply(lambda x: " / ".join(x))
df = df.drop(columns = ['Proveedor_siglas'])

In [423]:
# Paso 4: Eliminación de algunos registros sin datos, forzar formatos y creación de columna de costo final de envío neto

df = df[df['SKU'].notna() & df['Cargo por venta e impuestos (CLP)'].notna() & df['Ingresos por productos (CLP)'].notna()]

columnas_numericas = [
    'Unidades', 'Ingresos por productos (CLP)', 'Cargo por venta e impuestos (CLP)',
    'Ingresos por envío (CLP)', 'Costos de envío (CLP)'
]

for col in columnas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors = 'coerce')
    if col in ['Ingresos por envío (CLP)', 'Costos de envío (CLP)']:
        df[col] = df[col].fillna(0)
    if col in ['Ingresos por productos (CLP)', 'Cargo por venta e impuestos (CLP)',
               'Ingresos por envío (CLP)', 'Costos de envío (CLP)']:
        df[f"{col} Neto"] = df[col] / 1.19

df['Costo final envío (CLP) Neto'] = df.apply(lambda row: 3500 / 1.19 - row['Ingresos por envío (CLP) Neto'] if row['Forma de entrega'] == 'Mercado Envíos Flex' else - (row['Costos de envío (CLP) Neto'] + row['Ingresos por envío (CLP) Neto']), axis=1)

"""
Anteriormente, el paso 3 y 4 iban más adelante, posterior a la obtención de las bases de ingresos e ingresos sfe. Ahora se editó esto para que las bases recién mencionadas tengas las columnas de sku limpio y valores netos
"""

'\nAnteriormente, el paso 3 y 4 iban más adelante, posterior a la obtención de las bases de ingresos e ingresos sfe. Ahora se editó esto para que las bases recién mencionadas tengas las columnas de sku limpio y valores netos\n'

In [424]:
# Edición de base de datos para análisis de márgenes (df)

# Paso 4: Eliminación de columnas indeseadas

columnas_unicas = []
contador_estado = 0
contador_unidades = 0
contador_forma_entrega = 0
columnas_eliminar = [
        "Descripción del estado", "Paquete de varios productos", "Pertenece a un kit",
        "Anulaciones y reembolsos (CLP)", "Total (CLP)",
        "Precio unitario de venta de la publicación (CLP)",
        "Mes de facturación de tus cargos", "Venta por publicidad",
        "Canal de venta", "Tienda oficial", "Variante", "Tipo de publicación", "Factura adjunta",
        "Datos personales o de empresa", "Tipo y número de documento", "Dirección",
        "Tipo de contribuyente", "Actividad económica", "Comprador", "Negocio", "Cédula",
        "Domicilio", "Comuna", "Estado", "Código postal", "País", "Fecha en camino", "Fecha entregado",
        "Transportista", "Número de seguimiento", "URL de seguimiento",
        "Revisado por Mercado Libre", "Fecha de revisión",
        "Dinero a favor", "Resultado", "Destino", "Motivo del resultado",
        "Reclamo abierto", "Reclamo cerrado", "Con mediación"
    ]

for col in df.columns:
    if isinstance(col, str) and col.startswith("Estado"):
        contador_estado += 1
        if contador_estado == 1:
              columnas_unicas.append(col)
    elif isinstance(col, str) and col.startswith("Unidades"):
        contador_unidades += 1
        if contador_unidades == 1:
              columnas_unicas.append(col)
    elif isinstance(col, str) and col.startswith("Forma de entrega"):
        contador_forma_entrega += 1
        if contador_forma_entrega == 1:
            columnas_unicas.append(col)
    elif isinstance(col, str) and all(not col.startswith(nombre_col) for nombre_col in columnas_eliminar):
          columnas_unicas.append(col)

 
df = df[columnas_unicas]

In [425]:
# Paso 5: Filtrar estados

print(f"Existen {len(df)} registros previo al filtro de estados")

inicios_estados_deseados = (
    'Acuerdas la entrega',
    'Despacharemos el paquete',
    'El envío está demorado, pero ya tienes el dinero disponible',
    'En camino',
    'En punto de retiro',
    'Entregado',
    'Etiqueta lista para imprimir',
    'Etiqueta para imprimir',
    'Etiqueta impresa'
    'Envío demorado',
    'Envío reprogramado',
    'Listo para recolección',
    'Llega el',
    'Llega entre el',
    'Mediación finalizada. Te dimos el dinero',
    'Procesando en la bodega',
    'Venta concretada',
    'Venta entregada',
    'Venta no entregada. Te dimos el dinero'
)

df_sfe = df.copy()
df = df[df['Estado'].apply(lambda x: any(x.startswith(p) for p in inicios_estados_deseados))].reset_index(drop = True)
print(f"Luego de filtrar los estados, quedan {len(df)} registros con estados deseados (ventas)")

Existen 323 registros previo al filtro de estados
Luego de filtrar los estados, quedan 294 registros con estados deseados (ventas)


In [426]:
# Del paso 5 se obtienen los ingresos sin filtrar estado y los ingresos filtrando estado

output_path_ingresos_sin_filtrar = f'{output_folder}/{cuenta_meli} {fecha} Ingresos sin filtrar estado.xlsx'
df_sfe.to_excel(output_path_ingresos_sin_filtrar, index = False)

"""
La diferencia entre df_ingresos y df_sfe es que el primero contiene las ventas en paquete (no el encabezado, si no que cada una de las ventas), ya que se quiere hacer el análisis de ingresos considerando todos los registros.
Como el análisis de márgenes solo se realiza con las ventas unitarias, es decir, no se consideran las ventas en paquete, se genera df_sfe, que contiene todos los registros a excepción de las ventas en paquete.
"""

output_path_ingresos_filtrados = f'{output_folder}/{cuenta_meli} {fecha} Ingresos con estados filtrados.xlsx'
df.to_excel(output_path_ingresos_filtrados, index = False)

In [427]:
# Paso 7: Separación de SKUs

sku_expanded = df['SKU_limpio'].apply(expand_sku)
max_length = sku_expanded.apply(len).max()
sku_df = pd.DataFrame(sku_expanded.tolist(), columns=[f'SKU_{i+1}' for i in range(max_length)])
df = pd.concat([df, sku_df], axis=1)

In [428]:
# Paso 8: Cruzar costos full con df

archivo_costos = f'/Users/martincarrasco/Desktop/Martín_Carrasco/Reportes/2025/Cuentas RDS/COSTOS PARA CRUCE/Costos para cruce al {fecha_costos}.xlsx'
hoja_costos_full = 'CostosFull'
df_costos_full = pd.read_excel(archivo_costos, sheet_name = hoja_costos_full, usecols = ['SKU', 'PRECIO CONFIRMADO'])

# df_costos_full['SKU'] = df_costos_full['SKU'].astype(str).str.upper().str.strip()
df_costos_full['SKU'] = df_costos_full['SKU'].apply(limpiar_sku)

duplicados_costos_full = df_costos_full.duplicated(subset=['SKU'], keep='first')
if duplicados_costos_full.any():
    df_costos_full = df_costos_full[~duplicados_costos_full]
    print(f"Se eliminaron {duplicados_costos_full.sum()} filas duplicadas en el DataFrame de costos.")

sku_a_costo_full = df_costos_full.set_index('SKU')['PRECIO CONFIRMADO'].to_dict()

sku_cols = [col for col in df.columns if re.match(r"SKU_\d+$", col)]

for col in sku_cols:
    df[f'Costo_full_{col}'] = df[col].map(sku_a_costo_full)

# for col in sku_cols:
#     df = df.merge(
#         df_costos_full.rename(columns={
#             "SKU": col,
#             "PRECIO CONFIRMADO": f"Costo_full_{col}"
#         }),
#         on=col,
#         how="left"
#     )

Se eliminaron 212 filas duplicadas en el DataFrame de costos.


In [429]:
# Paso 9: Cruzar costos normales con df

hoja_costos = 'Costos'
df_costos = pd.read_excel(archivo_costos, sheet_name = hoja_costos, usecols = ['SKU', 'PRECIO'])

# df_costos['SKU'] = df_costos['SKU'].astype(str).str.upper().str.strip()
df_costos['SKU'] = df_costos['SKU'].apply(limpiar_sku)

duplicados_costos = df_costos.duplicated(subset=['SKU'], keep='first')
print(f'cantidad de costos duplicados: {duplicados_costos.sum()}')
if duplicados_costos.any():
    df_costos = df_costos[~duplicados_costos]
    print(f"Se eliminaron {duplicados_costos.sum()} filas duplicadas en el DataFrame de costos.")

sku_a_costo = df_costos.set_index('SKU')['PRECIO'].to_dict()

for col in sku_cols:
    df[f'Costo_{col}'] = df[col].map(sku_a_costo)

# for col in sku_cols:
#     df = df.merge(
#         df_costos.rename(columns={
#             "SKU": col,
#             "PRECIO": f"Costo_{col}"
#         }),
#         on=col,
#         how="left"
#     )

cantidad de costos duplicados: 127
Se eliminaron 127 filas duplicadas en el DataFrame de costos.


In [430]:
# Paso 10: Aplicar descuentos cuando corresponda

fecha_inicio_descuentos = pd.Timestamp('2025-09-01')
fecha_fin_descuentos = pd.Timestamp('2025-09-08')

reglas_adicionales = [
    lambda row: descuento_sku_prefijo(
        row,
        prefijo='CR- ',
        porcentaje=0.03,
        fecha_col='Fecha de venta',
        fecha_inicio=pd.Timestamp('2025-07-28'), # Personalizar en caso de querer añadir otra regla
        fecha_fin=pd.Timestamp('2025-07-31'), # Personalizar en caso de quere añadir otra regla
        excluir_full = True
    )
]

df = aplicar_descuentos(
    df,
    fecha_col = 'Fecha de venta',
    fecha_inicio = fecha_inicio_descuentos,
    fecha_fin = fecha_fin_descuentos,
    descuentos_dict = descuentos_importadoras,
    activar = ACTIVAR_DESCUENTOS,
    reglas_extra = reglas_adicionales # Recordar añadir reglas_extra = reglas_adicionales para añadir cualquier regla adicional que se desee
)

df['Costo_final_producto'] = df.apply(calcular_costo_final, axis = 1) * df['Unidades']

Descuentos no aplicados, devolviendo el DataFrame original.


In [431]:
# Paso 11: Eliminar ventas a las que no se les haya encontrado costos a sus productos asociados

columnas_sku = [col for col in df.columns if col.startswith("SKU_") and col[-1].isdigit()]

df['SKU_faltante'] = df.apply(lambda row: detectar_sku_faltante(row, columnas_sku), axis = 1)

df = df[df['SKU_faltante'] == 'Sin SKU faltante']
print(f'Eliminando los registros con algún SKU sin costo, quedan {len(df)} registros para analizar')

Eliminando los registros con algún SKU sin costo, quedan 284 registros para analizar


In [432]:
# Paso 12: Cálculo de márgenes

df['Rango de precio'] = df['Ingresos por productos (CLP)'].apply(clasificar_precio)

df['Total costo'] = (
        df["Costo_final_producto"] -
        df["Cargo por venta e impuestos (CLP) Neto"] +
        df["Costo final envío (CLP) Neto"]
    )
df["Utilidad"] = df["Ingresos por productos (CLP) Neto"] - df["Total costo"]
df["Margen"] = df["Utilidad"] / df["Ingresos por productos (CLP) Neto"]

condiciones_pricing = [
    df['SKU_MAYUSC'].str.contains('XX- ', na = False),
    df['SKU_MAYUSC'].str.contains('Z- ', na = False)
]

opciones_pricing = [
    'Killer',
    'Liquidación'
]

df['Estrategia Pricing'] = np.select(condiciones_pricing, opciones_pricing, default = 'Normal')

utilidad_total = df["Utilidad"].sum()
df["Ponderado"] = df["Utilidad"] / utilidad_total
df["Margen x Ponderado"] = df["Margen"] * df["Ponderado"]

df["Cantidad SKUs"] = df[columnas_sku].notna().sum(axis=1)

print(f'Considerando todos los registros con venta, en donde a todos los sku asociados se les haya encontrado costo, para la cuenta {cuenta_meli} en {fecha}, el margen promedio simple es {df["Margen"].mean() * 100:.2f}%')
print(f'Considerando todos los registros con venta, en donde a todos los sku asociados se les haya encontrado costo, para la cuenta {cuenta_meli} en {fecha}, el margen promedio ponderado es {df["Margen x Ponderado"].sum() * 100:.2f}%')

Considerando todos los registros con venta, en donde a todos los sku asociados se les haya encontrado costo, para la cuenta TYC en AGOSTO 2025, el margen promedio simple es 15.54%
Considerando todos los registros con venta, en donde a todos los sku asociados se les haya encontrado costo, para la cuenta TYC en AGOSTO 2025, el margen promedio ponderado es 15.37%


In [433]:
# Del paso 12 se obtiene la base de datos final con los márgenes

output_path_margenes = f'{output_folder}/{cuenta_meli} {fecha} Márgenes.xlsx'
with pd.ExcelWriter(f'{output_path_margenes}', engine = 'openpyxl') as writer:
    df.to_excel(writer, sheet_name = 'Márgenes', index = False)